## Section 1 - Environment Setup and Dependency Installation

In [ ]:
!pip install unsloth -q

## Section 2 — Library Imports, Experimental Reproducibility, and Configuration


In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth import UnslothTrainer, UnslothTrainingArguments
from unsloth.chat_templates import train_on_responses_only

import os
import re
import gc
import json
import random
import warnings

from decimal import Decimal, InvalidOperation

import numpy as np
import pandas as pd
import torch
import transformers
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from google.colab import drive
from tqdm.auto import tqdm
from datasets import Dataset
from transformers import EarlyStoppingCallback, TrainerCallback

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
drive.mount("/content/drive")

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Reproducibility Seed Setting

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Main Folder & Path Setting

In [ ]:
MODEL_MATH = "Qwen/Qwen2.5-Math-1.5B-Instruct"

# Model SLERP FINAL dari Kode 03 (alpha=0.0840, EN 84.69% / ID 45.11%)
SLERP_MERGED_DIR = (
    "/content/drive/MyDrive/Dataset Skripsi/"
    "Result Model Merging Dual Prompt Full Size/final_slerp_mergekit"
)

# Ringkasan hasil SLERP pre-SFT (dari Kode 03) -- dipakai sebagai baseline pembanding
SLERP_BASELINE_SUMMARY_PATH = (
    "/content/drive/MyDrive/Dataset Skripsi/"
    "Result Model Merging Dual Prompt Full Size/evaluation/final_slerp_summary.csv"
)

# Dataset SFT full (~7,473 soal) -- GANTI path sesuai lokasi file training-mu
SFT_DATASET_PATH_4K = (
    "/content/drive/MyDrive/Dataset Skripsi/dataset/sft/sft_dataset.json"
)
SFT_DATASET_PATH_FULL = (
    "/content/drive/MyDrive/Dataset Skripsi/dataset/sft/sft_qwen_merged_dataset.json"
)

# Test set (sama persis dengan yang dipakai baseline & SLERP)
TEST_DATASET_PATH = (
    "/content/drive/MyDrive/Dataset Skripsi/dataset/test/"
    "Full Size Test Set - V3/GSM8K Test Set.json"
)

ROOT_OUTPUT_DIR = "/content/drive/MyDrive/Dataset Skripsi/Result SFT Full Training"
EVAL_DIR = os.path.join(ROOT_OUTPUT_DIR, "evaluation")
CHARTS_DIR = os.path.join(ROOT_OUTPUT_DIR, "charts")
RUNS_DIR = os.path.join(ROOT_OUTPUT_DIR, "runs")

# Ringkasan gabungan (pure vs merged) akan disimpan di sini pada Section 14.
COMBINED_SFT_SUMMARY_PATH = os.path.join(EVAL_DIR, "pure_vs_merged_sft_summary.csv")

for directory in [ROOT_OUTPUT_DIR, EVAL_DIR, CHARTS_DIR, RUNS_DIR]:
    os.makedirs(directory, exist_ok=True)

### Inference and Chart Static Variables

In [ ]:
STRATA = ["L1", "L2", "L3"]

# Palet warna diseragamkan dengan chart dual-prompt & SLERP (Kode 02/03):
# peach untuk Bahasa Inggris, oranye tua untuk Bahasa Indonesia.
EN_COLOR = "#F5CBA7"
ID_COLOR = "#E67E22"

# Palet tambahan khusus kurva training (train vs eval), tetap satu keluarga
# dengan warna yang sudah dipakai sepanjang hari ini (biru dari chart single-
# prompt, oranye dari chart dual-prompt/SLERP) supaya seluruh rangkaian chart
# skripsi terasa satu sistem visual.
TRAIN_COLOR = "#4A90D9"
EVAL_COLOR = "#E67E22"

MAX_SEQ_LENGTH = 2048  # konsisten di training & inference
LOAD_IN_4BIT = False

GENERATION_CONFIG = {
    "max_new_tokens": 1024,
    "do_sample": False,
    "use_cache": True,
}

# Ukuran sampel untuk tracking akurasi downstream PER EPOCH (bukan full test
# set -- full-set setiap epoch terlalu mahal waktu). Full test set tetap
# dipakai untuk evaluasi AKHIR pasca-training di Section 11-12.
EPOCH_EVAL_SAMPLE_PER_STRATUM = 40  # ~120 soal/bahasa, cepat tapi representatif

sns.set_theme(style="whitegrid", rc={
    "axes.facecolor": "#F8F9FA",
    "figure.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Section 3 - Utils & Data Preprocessing

In [ ]:
def normalize_number(text):

    if text is None:
        return None

    text = str(text).strip()

    text = text.replace("$", "")
    text = text.replace("Rp.", "")
    text = text.replace("Rp", "")
    text = text.replace("\\,", "")
    text = text.replace(",", "")
    text = text.replace("%", "")
    text = text.strip().rstrip(".")

    text = re.sub(r"(?i)\b(orang|buah|kali|rupiah|unit|hari|jam|menit)\b", "", text)
    text = text.strip()

    if re.fullmatch(r"-?\d+/\d+", text):
        numerator, denominator = text.split("/")
        if denominator != "0":
            value = Decimal(numerator) / Decimal(denominator)
            return format(value.normalize(), "f")

    try:
        value = Decimal(text)
        return format(value.normalize(), "f")
    except (InvalidOperation, ValueError):
        return text.lower()


def extract_answer(text):
    """
    Prioritas extraction:
    1. \\boxed{...}
    4. #### ...
    5. Angka terakhir sebagai fallback
    """
    if text is None:
        return None, "missing"

    text = str(text).strip()

    patterns = [
        (r"\\boxed\{([^{}]+)\}", "boxed"),
        (r"(?im)^\s*####\s*([^\n]+)", "gsm8k_marker"),
    ]

    for pattern, method in patterns:
        matches = re.findall(pattern, text)

        valid_matches = [m.strip() for m in matches if m.strip() and re.search(r"\d", m)]
        if valid_matches:
            return valid_matches[-1], method

    numbers = re.findall(r"-?\d+(?:[.,]\d+)?(?:/\d+)?", text)
    if numbers:
        return numbers[-1].strip(), "last_number_fallback"

    return None, "not_found"


def is_correct(raw_output, ground_truth):
    prediction, extraction_method = extract_answer(raw_output)

    normalized_prediction = normalize_number(prediction)
    normalized_ground_truth = normalize_number(ground_truth)

    correct = (
        normalized_prediction is not None
        and normalized_prediction == normalized_ground_truth
    )

    return {
        "prediction": prediction,
        "normalized_prediction": normalized_prediction,
        "normalized_ground_truth": normalized_ground_truth,
        "extraction_method": extraction_method,
        "correct": correct,
    }


def clear_vram(model=None, tokenizer=None, trainer=None):
    if trainer is not None:
        del trainer
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Section 4 - System Prompts

In [ ]:
SYSTEM_PROMPT_EN = r"""
You are a mathematical reasoning assistant.

Solve the problem carefully, step by step.
On the final line, write only the final numerical answer in this format:
\boxed{value}
"""

SYSTEM_PROMPT_ID = r"""
Anda adalah asisten penalaran matematika.

Selesaikan soal dengan cermat secara bertahap.
Pada baris terakhir, tuliskan hanya jawaban numerik akhir dengan format:
\boxed{nilai}
"""

## Section 5 - Dataset Loading & Validation

### Test Set

In [ ]:
with open(TEST_DATASET_PATH, "r", encoding="utf-8") as f:
    raw_test = json.load(f)

required_fields_test = {"question", "translated_question", "answer", "stratum"}

for idx, item in enumerate(raw_test):
    missing = required_fields_test - set(item.keys())
    if missing:
        raise ValueError(f"Data test indeks {idx} tidak memiliki field: {missing}")

df_test = pd.DataFrame(raw_test)
df_test["question"] = df_test["question"].astype(str).str.strip()
df_test["translated_question"] = df_test["translated_question"].astype(str).str.strip()
df_test["answer"] = df_test["answer"].astype(str).str.strip()
df_test["stratum"] = df_test["stratum"].astype(str).str.strip()

if not set(df_test["stratum"].unique()).issubset(set(STRATA)):
    raise ValueError(f"Strata tidak valid: {sorted(df_test['stratum'].unique())}")

full_q_en = df_test["question"].tolist()
full_q_id = df_test["translated_question"].tolist()
full_answers = df_test["answer"].tolist()
full_strata = df_test["stratum"].tolist()

### SFT Dataset All Dataset

In [ ]:
def load_sft_dataset(path, cot_source="solution"):
    with open(path, "r", encoding="utf-8") as f:
        raw_sft = json.load(f)

    sft_records = raw_sft.items() if isinstance(raw_sft, dict) else enumerate(raw_sft)

    sft_list = []
    for key, item in sft_records:
        ans = str(item["answer"]).strip()
        question_id = str(item["translated_question"]).strip()

        if cot_source == "solution":
            solution = str(item["solution"]).strip()
            if r"\boxed{" not in solution:
                solution = f"{solution}\n\nJAWABAN: \\boxed{{{ans}}}"
        elif cot_source == "original_solution":
            solution = str(item["original_solution"]).strip()
            if r"\boxed{" not in solution:
                solution = solution.split("####")[0].strip()
                solution = f"{solution}\n\\boxed{{{ans}}}"
        else:
            raise ValueError(f"cot_source tidak dikenal: {cot_source}")

        entry = {
            "question": question_id,
            "answer": ans,
            "solution_distilled": solution,
            "stratum": item.get("stratum"),
        }
        if "question" in item:
            entry["question_en"] = str(item["question"]).strip()

        sft_list.append(entry)

    df_sft = pd.DataFrame(sft_list)

    return df_sft


### Validasi: Filter data train

In [ ]:
def prepare_train_eval_split(df_sft, df_test, seed=SEED, test_size=0.1):
    train_question_set = set(df_sft["question"].str.strip())
    test_question_set_id = set(df_test["translated_question"].str.strip())

    overlap = train_question_set & test_question_set_id
    if len(overlap) > 0:
        raise ValueError(
            f"Data leakage terdeteksi: {len(overlap)} soal SFT overlap dengan test set!\n"
            f"Contoh: {list(overlap)[:3]}"
        )
    print(f"Validasi leakage: OK ({len(df_sft)} soal SFT bersih).")

    hf_dataset = Dataset.from_pandas(df_sft)
    hf_dataset = hf_dataset.train_test_split(test_size=test_size, seed=seed)

    train_dataset = hf_dataset["train"]
    eval_dataset = hf_dataset["test"]

    print(f"SFT Train : {len(train_dataset)}")
    print(f"SFT Eval  : {len(eval_dataset)}")

    return train_dataset, eval_dataset

### Train/Eval split untuk monitoring training

In [ ]:
df_sft = load_sft_dataset(SFT_DATASET_PATH_4K, cot_source="solution")
hf_dataset = Dataset.from_pandas(df_sft)
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=SEED)

train_dataset = hf_dataset["train"]
eval_dataset = hf_dataset["test"]

print(f"SFT Train : {len(train_dataset)}")
print(f"SFT Eval  : {len(eval_dataset)}")
print(f"Test EN   : {len(full_q_en)}")
print(f"Test ID   : {len(full_q_id)}")

df_eval_split = eval_dataset.to_pandas()

epoch_val_q_id = df_eval_split["question"].tolist()
epoch_val_answers = df_eval_split["answer"].tolist()

EPOCH_TRACK_EN_AVAILABLE = "question_en" in df_eval_split.columns

if EPOCH_TRACK_EN_AVAILABLE:
    epoch_val_q_en = df_eval_split["question_en"].tolist()
else:
    epoch_val_q_en = None
    print(
        "Catatan: kolom 'question_en' tidak tersedia di data SFT -- "
        "tracking EN per-epoch akan di-skip (bukan diganti sampel test set)."
    )

print(f"Sampel tracking per-epoch (dari eval_dataset SFT): {len(epoch_val_q_id)} soal")

SFT Train : 3442
SFT Eval  : 383
Test EN   : 1319
Test ID   : 1319
Sampel tracking per-epoch (dari eval_dataset SFT): 383 soal


## Section 6 - Audit Panjang Token

In [ ]:
def audit_completion_length(dataset, tokenizer, column, max_seq_length):
    lengths = []
    for text in dataset[column]:
        lengths.append(len(tokenizer.encode(text)))

    lengths = np.array(lengths)
    over_limit = (lengths > max_seq_length * 0.9).sum()

    print(f"\nAudit panjang token kolom '{column}':")
    print(f"  Rata-rata      : {lengths.mean():.1f} token")
    print(f"  Maksimum       : {lengths.max()} token")
    print(f"  P95            : {np.percentile(lengths, 95):.1f} token")
    print(f"  Berisiko terpotong (>{int(max_seq_length*0.9)} token): {over_limit} dari {len(lengths)}")

    return lengths


## Section 7 - Inference & Evaluation Engine

In [ ]:
def run_inference(model, tokenizer, questions, system_prompt, batch_size=16, desc="Inferensi"):
    model.eval()

    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    all_outputs = []

    try:
        for start_idx in tqdm(range(0, len(questions), batch_size), desc=desc):
            batch_questions = questions[start_idx : start_idx + batch_size]

            batch_texts = [
                tokenizer.apply_chat_template(
                    [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": q.strip()},
                    ],
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for q in batch_questions
            ]

            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            ).to(model.device)

            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=GENERATION_CONFIG["max_new_tokens"],
                    do_sample=False,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id,
                    use_cache=True,
                )

            prompt_length = inputs.input_ids.shape[1]
            generated_tokens = outputs[:, prompt_length:]

            decoded_outputs = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
            all_outputs.extend(output.strip() for output in decoded_outputs)

            del inputs, outputs, generated_tokens
    finally:
        # Selalu dikembalikan, bahkan kalau terjadi error di tengah loop --
        # supaya training yang dilanjutkan setelah callback tidak mewarisi
        # padding_side yang salah.
        tokenizer.padding_side = original_padding_side
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return all_outputs

In [ ]:
def evaluate_and_save(questions, raw_outputs, ground_truths, strata_list, output_prefix,
                       language, model_name, prompt_name, save_files=True):
    records = []

    for question, raw_output, answer, stratum in zip(questions, raw_outputs, ground_truths, strata_list):
        result = is_correct(raw_output, answer)
        records.append({
            "model": model_name,
            "language": language,
            "prompt_name": prompt_name,
            "question": question,
            "raw_output": raw_output,
            "extracted_answer": result["prediction"],
            "normalized_prediction": result["normalized_prediction"],
            "ground_truth": answer,
            "normalized_ground_truth": result["normalized_ground_truth"],
            "extraction_method": result["extraction_method"],
            "stratum": stratum,
            "correct": result["correct"],
        })

    df_results = pd.DataFrame(records)

    if save_files:
        df_results.to_json(f"{output_prefix}.jsonl", orient="records", lines=True, force_ascii=False)
        df_results.to_excel(f"{output_prefix}.xlsx", index=False)
        df_results.to_csv(f"{output_prefix}.csv", index=False, encoding="utf-8-sig")

    accuracy = df_results["correct"].mean()
    per_stratum = df_results.groupby("stratum")["correct"].mean().reindex(STRATA, fill_value=0.0).to_dict()
    extraction_stats = df_results["extraction_method"].value_counts(normalize=True).mul(100).round(2).to_dict()

    if save_files:
        print(f"\nModel        : {model_name}")
        print(f"Bahasa         : {language}")
        print(f"Prompt         : {prompt_name}")
        print(f"Akurasi        : {accuracy * 100:.2f}%")
        print("Akurasi strata :")
        for stratum, score in per_stratum.items():
            print(f"  {stratum}: {score * 100:.2f}%")
        print("Metode ekstraksi jawaban (%):")
        print(extraction_stats)

    return {
        "accuracy": accuracy,
        "per_stratum": per_stratum,
        "extraction_stats": extraction_stats,
        "dataframe": df_results,
    }

In [ ]:
def quick_accuracy(model, tokenizer, questions, answers, system_prompt, desc):
    """Versi ringan untuk tracking per-epoch -- tidak menyimpan file ke disk."""
    raw_outputs = run_inference(model, tokenizer, questions, system_prompt, desc=desc)
    correct = sum(is_correct(raw, gt)["correct"] for raw, gt in zip(raw_outputs, answers))
    return correct / len(answers) if answers else 0.0

## Section 8 - Callback: Tracking Akurasi Downstream Per Epoch (sampel stratifikasi)

In [ ]:
class EpochDownstreamEvalCallback(TrainerCallback):
    def __init__(self, model, tokenizer, run_name, val_q_id, val_answers, val_q_en=None):
        self.model = model
        self.tokenizer = tokenizer
        self.run_name = run_name
        self.val_q_id = val_q_id
        self.val_answers = val_answers
        self.val_q_en = val_q_en
        self.records = []

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch_num = int(round(state.epoch))
        print(f"\n[Epoch {epoch_num}] Menjalankan tracking akurasi (validation split SFT)...")

        try:
            FastLanguageModel.for_inference(self.model)

            # Evaluasi ID (cross-lingual) memakai SYSTEM_PROMPT_ID
            acc_id_cross = quick_accuracy(
                self.model, self.tokenizer, self.val_q_id, self.val_answers,
                SYSTEM_PROMPT_ID, desc=f"{self.run_name} | Epoch {epoch_num} | ID (val)",
            )
            record = {"epoch": epoch_num, "accuracy_id_cross_val": acc_id_cross}

            # Evaluasi EN tetap pakai SYSTEM_PROMPT_EN sebagai baseline
            if self.val_q_en is not None:
                acc_en = quick_accuracy(
                    self.model, self.tokenizer, self.val_q_en, self.val_answers,
                    SYSTEM_PROMPT_EN, desc=f"{self.run_name} | Epoch {epoch_num} | EN (val)",
                )
                record["accuracy_en_val"] = acc_en
                print(f"[Epoch {epoch_num}] EN(val)={acc_en*100:.2f}%  ID(val)={acc_id_cross*100:.2f}%")
            else:
                print(f"[Epoch {epoch_num}] ID(val)={acc_id_cross*100:.2f}%  (EN val: tidak tersedia)")

            self.records.append(record)
        except Exception as e:
            print(f"[Epoch {epoch_num}] PERINGATAN: evaluasi per-epoch gagal ({e}), training dilanjutkan.")
        finally:
            FastLanguageModel.for_training(self.model)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return control


## Section 9 - Visualisasi Engine (SERAGAM dengan Kode 02 & 03)




### Chart 1: Komparasi Akurasi EN vs ID per Skenario

In [ ]:
def plot_bilingual_comparison(scenario_labels, acc_en_list, acc_id_list, title, filename, id_label="Bahasa Indonesia"):
    # 1. Menyiapkan Data Dinamis
    labels = scenario_labels
    inggris_scores = [acc * 100 for acc in acc_en_list]
    indonesia_scores = [acc * 100 for acc in acc_id_list]

    # Simpan untuk return DataFrame agar tidak merusak alur di luar fungsi
    data = []
    for label, acc_en, acc_id in zip(scenario_labels, acc_en_list, acc_id_list):
        data.append({"Skenario": label, "Bahasa": "Bahasa Inggris", "Akurasi": acc_en * 100})
        data.append({"Skenario": label, "Bahasa": id_label, "Akurasi": acc_id * 100})
    df_plot = pd.DataFrame(data)

    x = np.arange(len(labels))

    # 2. Pengaturan Layout Bar
    width = 0.34
    offset = 0.19

    # 3. Palet Warna Baku
    color_en = '#f2c79e'    # Oranye terang
    color_id = '#d2691e'    # Oranye gelap
    text_color = '#777777'
    title_color = '#666666'

    # 4. Membuat Figure dan Axes (Ukuran 11, 6)
    fig, ax = plt.subplots(figsize=(11, 6), facecolor='white')

    # 5. Membuat Bar Chart
    rects1 = ax.bar(x - offset, inggris_scores, width, label='Bahasa Inggris', color=color_en)
    # Gunakan parameter id_label dinamis untuk label bahasa kedua
    rects2 = ax.bar(x + offset, indonesia_scores, width, label=id_label, color=color_id)

    # 6. Kustomisasi Sumbu (Spines & Ticks)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')

    # Pengaturan Sumbu Y
    ax.set_ylim(0, 110)
    ax.set_yticks(np.arange(0, 120, 20))
    ax.set_ylabel('Akurasi (%)', color=text_color, fontsize=12, labelpad=12)
    ax.tick_params(axis='y', colors=text_color)

    # Pengaturan Sumbu X (Label bawah dikosongkan sesuai request asli Anda)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, color=text_color, fontsize=11)
    ax.set_xlabel('', color=text_color, fontsize=12)
    ax.tick_params(axis='x', colors=text_color)

    # 7. Menambahkan Label Nilai di Atas Bar
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.1f}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 4),
                        textcoords="offset points",
                        ha='center', va='bottom',
                        fontsize=10, color='black')

    autolabel(rects1)
    autolabel(rects2)

    # 8. Menambahkan Judul (Single line, ditaruh di tengah atas)
    fig.text(0.5, 0.94, title, ha='center', va='center',
             fontsize=17, fontweight='bold', color=title_color)

    # 9. Menambahkan Legend
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=2, frameon=False,
              handlelength=1, handleheight=1, labelcolor=text_color, fontsize=11)

    # 10. Mengatur Layout, Menyimpan, dan Menampilkan Chart
    plt.tight_layout()
    # Margin sedikit lebih rapat dibanding chart yang pakai subjudul
    plt.subplots_adjust(top=0.82)

    # Blok try-except menggunakan variabel CHARTS_DIR
    try:
        save_path = os.path.join(CHARTS_DIR, filename)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    except NameError:
        print(f"Info: Gambar belum disimpan karena CHARTS_DIR tidak ditemukan.")

    plt.show()

    return df_plot

### Chart 2: Akurasi per Strata

In [ ]:
def plot_stratum(per_stratum_en, per_stratum_id, title, filename, id_label="Bahasa Indonesia"):
    # 1. Menyiapkan Data Dinamis dari Dictionary berdasarkan STRATA
    labels = []
    inggris_scores = []
    indonesia_scores = []
    data = []

    # Asumsi variabel global STRATA sudah didefinisikan sebelumnya
    for stratum in STRATA:
        acc_en = per_stratum_en.get(stratum, 0) * 100
        acc_id = per_stratum_id.get(stratum, 0) * 100

        labels.append(stratum)
        inggris_scores.append(acc_en)
        indonesia_scores.append(acc_id)

        # Simpan untuk return DataFrame agar alur fungsi tidak terputus
        data.append({"Strata": stratum, "Bahasa": "Bahasa Inggris", "Akurasi": acc_en})
        data.append({"Strata": stratum, "Bahasa": id_label, "Akurasi": acc_id})

    df_plot = pd.DataFrame(data)
    x = np.arange(len(labels))

    # 2. Pengaturan Layout Bar (Standar konsisten)
    width = 0.34
    offset = 0.19

    # 3. Palet Warna Baku
    color_en = '#f2c79e'    # Oranye terang
    color_id = '#d2691e'    # Oranye gelap
    text_color = '#777777'
    title_color = '#666666'

    # 4. Membuat Figure dan Axes (Ukuran 11, 6)
    fig, ax = plt.subplots(figsize=(11, 6), facecolor='white')

    # 5. Membuat Bar Chart
    rects1 = ax.bar(x - offset, inggris_scores, width, label='Bahasa Inggris', color=color_en)
    rects2 = ax.bar(x + offset, indonesia_scores, width, label=id_label, color=color_id)

    # 6. Kustomisasi Sumbu (Spines & Ticks)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')

    # Pengaturan Sumbu Y
    ax.set_ylim(0, 110)
    ax.set_yticks(np.arange(0, 120, 20))
    ax.set_ylabel('Akurasi (%)', color=text_color, fontsize=12, labelpad=12)
    ax.tick_params(axis='y', colors=text_color)

    # Pengaturan Sumbu X
    ax.set_xticks(x)
    ax.set_xticklabels(labels, color=text_color, fontsize=11)
    ax.set_xlabel('Strata', color=text_color, fontsize=12, labelpad=12)
    ax.tick_params(axis='x', colors=text_color)

    # 7. Menambahkan Label Nilai di Atas Bar
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.1f}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 4),
                        textcoords="offset points",
                        ha='center', va='bottom',
                        fontsize=10, color='black')

    autolabel(rects1)
    autolabel(rects2)

    # 8. Menambahkan Judul (Single line)
    fig.text(0.5, 0.94, title, ha='center', va='center',
             fontsize=17, fontweight='bold', color=title_color)

    # 9. Menambahkan Legend
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=2, frameon=False,
              handlelength=1, handleheight=1, labelcolor=text_color, fontsize=11)

    # 10. Mengatur Layout, Menyimpan, dan Menampilkan Chart
    plt.tight_layout()
    plt.subplots_adjust(top=0.82)

    # Blok try-except menggunakan variabel CHARTS_DIR
    try:
        save_path = os.path.join(CHARTS_DIR, filename)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    except NameError:
        print(f"Info: Gambar belum disimpan karena CHARTS_DIR tidak ditemukan.")

    plt.show()

    return df_plot

### Chart 3: Kurva Training & Eval Loss per Step

In [ ]:
def plot_training_curves(log_history, run_name):
    # 1. Ekstraksi Data
    train_records = [(l["step"], l["loss"]) for l in log_history if "loss" in l and "step" in l]
    eval_records = [(l["step"], l["eval_loss"]) for l in log_history if "eval_loss" in l and "step" in l]

    if not train_records and not eval_records:
        print(f"[{run_name}] Tidak ada log_history untuk diplot.")
        return

    # 2. Palet Warna Baku (Diadaptasi untuk Line Chart)
    color_train = '#f2c79e'  # Oranye terang (Training)
    color_eval = '#d2691e'   # Oranye gelap (Eval)
    text_color = '#777777'
    title_color = '#666666'

    # 3. Membuat Figure dan Axes (Ukuran konsisten 11, 6)
    fig, ax = plt.subplots(figsize=(11, 6), facecolor='white')

    # 4. Membuat Line Plot
    if train_records:
        steps, losses = zip(*train_records)
        ax.plot(steps, losses, color=color_train, linewidth=2.5,
                label="Training Loss", marker="o", markersize=5)

    if eval_records:
        steps, losses = zip(*eval_records)
        ax.plot(steps, losses, color=color_eval, linewidth=2.5,
                label="Eval Loss", marker="s", markersize=5)

    # 5. Kustomisasi Sumbu (Spines & Ticks)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')

    # Pengaturan Label Sumbu X dan Y
    ax.set_xlabel("Training Step", color=text_color, fontsize=12, labelpad=12)
    ax.set_ylabel("Loss", color=text_color, fontsize=12, labelpad=12)
    ax.tick_params(axis='x', colors=text_color, labelsize=11)
    ax.tick_params(axis='y', colors=text_color, labelsize=11)

    # 6. Menambahkan Judul dan Subjudul
    fig.text(0.5, 0.96, 'Kurva Training & Eval Loss', ha='center', va='center',
             fontsize=17, fontweight='bold', color=title_color)
    fig.text(0.5, 0.90, run_name, ha='center', va='center',
             fontsize=12, color=text_color)

    # 7. Menambahkan Legend
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=2, frameon=False,
              handlelength=1.5, handleheight=1, labelcolor=text_color, fontsize=11)

    # 8. Mengatur Layout, Menyimpan, dan Menampilkan Chart
    plt.tight_layout()
    plt.subplots_adjust(top=0.78)

    # Blok try-except menggunakan variabel CHARTS_DIR
    try:
        save_path = os.path.join(CHARTS_DIR, f"loss_curve_{run_name}.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    except NameError:
        print(f"Info: Gambar belum disimpan karena CHARTS_DIR tidak ditemukan.")

    plt.show()

### Chart 4: Progres Akurasi Downstream per Epoch

In [ ]:
def plot_epoch_accuracy_progression(epoch_records, run_name):
    if not epoch_records:
        print(f"[{run_name}] Tidak ada data epoch-eval untuk diplot.")
        return

    df_epoch = pd.DataFrame(epoch_records).sort_values("epoch")
    has_en = "accuracy_en_val" in df_epoch.columns

    # 1. Palet Warna Baku
    color_en = '#f2c79e'    # Oranye terang
    color_id = '#d2691e'    # Oranye gelap
    text_color = '#777777'
    title_color = '#666666'

    # 2. Membuat Figure dan Axes (Ukuran konsisten 11, 6)
    fig, ax = plt.subplots(figsize=(11, 6), facecolor='white')

    # 3. Membuat Line Plot & Label
    if has_en:
        ax.plot(df_epoch["epoch"], df_epoch["accuracy_en_val"] * 100,
                color=color_en, linewidth=2.5, marker="o", markersize=7,
                label="Bahasa Inggris (validation split)")

        for _, row in df_epoch.iterrows():
            ax.annotate(f"{row['accuracy_en_val']*100:.1f}%",
                        (row["epoch"], row["accuracy_en_val"] * 100),
                        textcoords="offset points", xytext=(0, 10),
                        ha="center", fontsize=9, fontweight="bold", color='black')

    ax.plot(df_epoch["epoch"], df_epoch["accuracy_id_cross_val"] * 100,
            color=color_id, linewidth=2.5, marker="o", markersize=7,
            label="Bahasa Indonesia Cross-lingual (validation split)")

    for _, row in df_epoch.iterrows():
        # Offset negatif jika ada bahasa Inggris agar label tidak saling tumpuk
        offset = -16 if has_en else 10
        ax.annotate(f"{row['accuracy_id_cross_val']*100:.1f}%",
                    (row["epoch"], row["accuracy_id_cross_val"] * 100),
                    textcoords="offset points", xytext=(0, offset),
                    ha="center", fontsize=9, fontweight="bold", color='black')

    # 4. Kustomisasi Sumbu (Spines & Ticks)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')

    ax.set_ylim(0, 110) # Beri sedikit ruang ekstra di atas (semula 100)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    ax.set_xlabel("Epoch", color=text_color, fontsize=12, labelpad=12)
    ax.set_ylabel("Akurasi (%)", color=text_color, fontsize=12, labelpad=12)
    ax.tick_params(axis='x', colors=text_color, labelsize=11)
    ax.tick_params(axis='y', colors=text_color, labelsize=11)

    # 5. Menambahkan Judul dan Subjudul
    fig.text(0.5, 0.98, 'Progres Akurasi Downstream per Epoch', ha='center', va='center',
             fontsize=17, fontweight='bold', color=title_color)
    fig.text(0.5, 0.92, f'{run_name} | (validation split dari data SFT, BUKAN test set)',
             ha='center', va='center', fontsize=11, color=text_color)

    # 6. Menambahkan Legend
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False,
              handlelength=1.5, handleheight=1, labelcolor=text_color, fontsize=10)

    # 7. Mengatur Layout, Menyimpan, dan Menampilkan
    plt.tight_layout()
    plt.subplots_adjust(top=0.78)

    # Blok try-except menggunakan variabel CHARTS_DIR
    try:
        save_path = os.path.join(CHARTS_DIR, f"epoch_accuracy_{run_name}.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    except NameError:
        print(f"Info: Gambar belum disimpan karena CHARTS_DIR tidak ditemukan.")

    plt.show()



### Chart 5: Distribusi Metode Ekstraksi Jawaban

In [ ]:
def plot_extraction_method_distribution(df_before, df_after, label_before, label_after, title, filename):
    def get_dist(df):
        return df["extraction_method"].value_counts(normalize=True).mul(100)

    dist_before = get_dist(df_before).rename(label_before)
    dist_after = get_dist(df_after).rename(label_after)

    df_dist = pd.concat([dist_before, dist_after], axis=1).fillna(0).T

    fig, ax = plt.subplots(figsize=(9, 5.5))
    df_dist.plot(kind="bar", stacked=True, ax=ax, colormap="YlOrBr", width=0.5)

    ax.set_title(title, pad=15, fontweight="bold")
    ax.set_ylabel("Proporsi (%)")
    ax.set_xlabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(title="Metode Ekstraksi", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS_DIR, filename), dpi=300, bbox_inches="tight")
    plt.show()

## Section 10 - Modular SFT Pipeline (LoRA standar: LR 2e-4, rslora, bias eksplisit)

In [ ]:
def apply_exp2_template(examples, tokenizer, target_cot_col):
    texts = []
    for q, s in zip(examples["question"], examples[target_cot_col]):
        msg = [
            {"role": "system", "content": SYSTEM_PROMPT_ID},
            {"role": "user", "content": q},
            {"role": "assistant", "content": s},
        ]
        texts.append(
            tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        )
    return {"text": texts}

In [ ]:
def run_sft_pipeline(base_model_path, run_name, target_cot_col, train_dataset, eval_dataset,
                      epoch_val_q_id, epoch_val_answers, epoch_val_q_en=None):

    output_path = os.path.join(RUNS_DIR, run_name)
    os.makedirs(output_path, exist_ok=True)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model_path, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=LOAD_IN_4BIT,
    )

    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    audit_completion_length(train_dataset, tokenizer, target_cot_col, MAX_SEQ_LENGTH)

    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_rslora=True, use_gradient_checkpointing="unsloth", random_state=SEED,
    )

    train_data = train_dataset.map(lambda ex: apply_exp2_template(ex, tokenizer, target_cot_col), batched=True)
    eval_data = eval_dataset.map(lambda ex: apply_exp2_template(ex, tokenizer, target_cot_col), batched=True)

    training_args = UnslothTrainingArguments(
        output_dir=output_path,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=3e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=70,
        save_strategy="steps",
        save_steps=70,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=SEED,
        report_to="none",
        per_device_eval_batch_size=8,
        eval_accumulation_steps=4,
    )

    epoch_callback = EpochDownstreamEvalCallback(
        model, tokenizer, run_name, val_q_id=epoch_val_q_id, val_answers=epoch_val_answers, val_q_en=epoch_val_q_en,
    )

    trainer = UnslothTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_data, eval_dataset=eval_data,
        dataset_text_field="text", args=training_args,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=6, early_stopping_threshold=0.001)],
    )

    trainer = train_on_responses_only(
        trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n",
    )

    print(f"\nMelatih model [{run_name}]...")
    trainer.train()

    model.save_pretrained(f"{output_path}/lora_model")
    tokenizer.save_pretrained(f"{output_path}/lora_model")

    print("\nEvaluasi FINAL pada Full Test Set (pasca-SFT)...")
    FastLanguageModel.for_inference(model)

    raw_id_cross = run_inference(
        model, tokenizer, full_q_id, SYSTEM_PROMPT_ID,
        desc=f"{run_name} | Evaluasi Soal ID",
    )

    res_id_cross = evaluate_and_save(
        full_q_id, raw_id_cross, full_answers, full_strata,
        os.path.join(output_path, "eval_id_cross_lingual"),
        language="Bahasa Indonesia (Prompt ID)", model_name=run_name, prompt_name="SYSTEM_PROMPT_ID",
    )

    raw_en = run_inference(
        model, tokenizer, full_q_en, SYSTEM_PROMPT_EN,
        desc=f"{run_name} | Evaluasi Soal EN",
    )

    res_en = evaluate_and_save(
        full_q_en, raw_en, full_answers, full_strata,
        os.path.join(output_path, "eval_en"),
        language="Bahasa Inggris", model_name=run_name, prompt_name="SYSTEM_PROMPT_EN",
    )

    clear_vram(model, tokenizer, trainer)

    return {
        "run_name": run_name, "target_cot_col": target_cot_col, "accuracy_en": res_en["accuracy"],
        "accuracy_id_cross": res_id_cross["accuracy"], "per_stratum_en": res_en["per_stratum"],
        "per_stratum_id_cross": res_id_cross["per_stratum"], "df_id_cross": res_id_cross["dataframe"],
        "df_en": res_en["dataframe"], "epoch_records": epoch_callback.records, "log_history": trainer.state.log_history,
    }

## Section 11 - Eksperimen 1: SFT pada Model PURE (Qwen2.5-Math-1.5B-Instruct)

In [ ]:
final_results = []

In [ ]:
RUN_NAME = "SFT_Pure_QwenMath_1.5B_Instruct_PromptID"

result_pure_model = run_sft_pipeline(
    base_model_path=MODEL_MATH,
    run_name=RUN_NAME,
    target_cot_col="solution_distilled",
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    epoch_val_q_id=epoch_val_q_id,
    epoch_val_answers=epoch_val_answers,
    epoch_val_q_en=epoch_val_q_en if EPOCH_TRACK_EN_AVAILABLE else None,
)

final_results.append(result_pure_model)

==((====))==  Unsloth 2026.8.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Audit panjang token kolom 'solution_distilled':
  Rata-rata      : 290.6 token
  Maksimum       : 770 token
  P95            : 438.9 token
  Berisiko terpotong (>1843 token): 0 dari 3442


Map:   0%|          | 0/3442 [00:00<?, ? examples/s]

Map:   0%|          | 0/383 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/3442 [00:00<?, ? examples/s]

Map:   0%|          | 0/383 [00:00<?, ? examples/s]


Melatih model [SFT_Pure_QwenMath_1.5B_Instruct_PromptID]...
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
{'loss': '0.594', 'grad_norm': '0.8492', 'learning_rate': '2.591e-05', 'epoch': '0.1856'}
{'loss': '0.3622', 'grad_norm': '0.642', 'learning_rate': '2.944e-05', 'epoch': '0.3712'}
{'loss': '0.2848', 'grad_norm': '0.6061', 'learning_rate': '2.739e-05', 'epoch': '0.5568'}
{'eval_loss': '0.2359', 'eval_runtime': '8.321', 'eval_samples_per_second': '46.03', 'eval_steps_per_second': '5.769', 'epoch': '0.6497'}
{'loss': '0.2406', 'grad_norm': '0.5765', 'learning_rate': '2.405e-05', 'epoch': '0.7425'}
{'loss': '0.2294', 'grad_norm': '0.7816', 'learning_rate': '1.977e-05', 'epoch': '0.9281'}
{'loss': '0.21', 'grad_norm': '0.6867', 'learning_rate': '1.5e-05', 'epoch': '1.111'}
{'loss': '0.2038', 'grad_norm': '0.737', 'learning_rate': '1.023e-05', 'epoch': '1.297'}
{'eval_loss': '0.2039', 'eval_runtime': '

SFT_Pure_QwenMath_1.5B_Instruct_PromptID | Evaluasi Soal ID:   0%|          | 0/83 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Visualisasi Hasil Eksperimen 1 (Pure Model)

In [ ]:
plot_training_curves(result_pure_model["log_history"], result_pure_model["run_name"])

In [ ]:
plot_epoch_accuracy_progression(result_pure_model["epoch_records"], result_pure_model["run_name"])

In [ ]:
plot_bilingual_comparison(
    scenario_labels=[result_pure_model["run_name"]],
    acc_en_list=[result_pure_model["accuracy_en"]],
    acc_id_list=[result_pure_model["accuracy_id_cross"]],
    title="Akurasi EN vs ID (Cross-lingual) — SFT Pure Model (Full Test Set)",
    filename=f"bilingual_comparison_{result_pure_model['run_name']}.png",
    id_label="Bahasa Indonesia (Cross-lingual)",
)

In [ ]:
plot_stratum(
    per_stratum_en=result_pure_model["per_stratum_en"],
    per_stratum_id=result_pure_model["per_stratum_id_cross"],
    title=f"Akurasi per Strata — {result_pure_model['run_name']}",
    filename=f"stratum_{result_pure_model['run_name']}.png",
    id_label="Bahasa Indonesia (Cross-lingual)",
)

## Section 12 - Eksperimen 2: SFT pada Model Hasil SLERP Merging



In [ ]:
RUN_NAME_MERGED = "SFT_SLERP_Merged_QwenMath_1.5B_PromptID"

result_merged_model = run_sft_pipeline(
    base_model_path=SLERP_MERGED_DIR,
    run_name=RUN_NAME_MERGED,
    target_cot_col="solution_distilled",
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    epoch_val_q_id=epoch_val_q_id,
    epoch_val_answers=epoch_val_answers,
    epoch_val_q_en=epoch_val_q_en if EPOCH_TRACK_EN_AVAILABLE else None,
)

final_results.append(result_merged_model)

### Visualisasi Hasil Eksperimen 2 (Merged Model)

In [ ]:
plot_training_curves(result_merged_model["log_history"], result_merged_model["run_name"])

In [ ]:
plot_epoch_accuracy_progression(result_merged_model["epoch_records"], result_merged_model["run_name"])

In [ ]:
plot_bilingual_comparison(
    scenario_labels=[result_merged_model["run_name"]],
    acc_en_list=[result_merged_model["accuracy_en"]],
    acc_id_list=[result_merged_model["accuracy_id_cross"]],
    title="Akurasi EN vs ID (Cross-lingual) — SFT Merged Model (Full Test Set)",
    filename=f"bilingual_comparison_{result_merged_model['run_name']}.png",
    id_label="Bahasa Indonesia (Cross-lingual)",
)

In [ ]:
plot_stratum(
    per_stratum_en=result_merged_model["per_stratum_en"],
    per_stratum_id=result_merged_model["per_stratum_id_cross"],
    title=f"Akurasi per Strata — {result_merged_model['run_name']}",
    filename=f"stratum_{result_merged_model['run_name']}.png",
    id_label="Bahasa Indonesia (Cross-lingual)",
)

## Section 13 - Perbandingan Langsung: SFT Pure Model vs SFT Merged Model

In [ ]:
mparison_scenario_labels = ["SFT: Pure Model", "SFT: Merged Model (SLERP)"]
comparison_acc_en = [result_pure_model["accuracy_en"], result_merged_model["accuracy_en"]]
comparison_acc_id = [result_pure_model["accuracy_id_cross"], result_merged_model["accuracy_id_cross"]]

plot_bilingual_comparison(
    scenario_labels=comparison_scenario_labels,
    acc_en_list=comparison_acc_en,
    acc_id_list=comparison_acc_id,
    title="Perbandingan Akurasi Pasca-SFT: Pure Model vs Merged Model (Full Test Set)",
    filename="comparison_pure_vs_merged_bilingual.png",
    id_label="Bahasa Indonesia (Cross-lingual)",
)

### Perbandingan Akurasi per Strata (Pure vs Merged, Bahasa Indonesia Cross-lingual)

In [ ]:
comparison_stratum_data = []

for stratum in STRATA:
    comparison_stratum_data.append({
        "Strata": stratum, "Skenario": "SFT: Pure Model",
        "Akurasi": result_pure_model["per_stratum_id_cross"].get(stratum, 0) * 100,
    })
    comparison_stratum_data.append({
        "Strata": stratum, "Skenario": "SFT: Merged Model (SLERP)",
        "Akurasi": result_merged_model["per_stratum_id_cross"].get(stratum, 0) * 100,
    })

df_comparison_stratum = pd.DataFrame(comparison_stratum_data)

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=df_comparison_stratum, x="Strata", y="Akurasi", hue="Skenario",
    palette={"SFT: Pure Model": EN_COLOR, "SFT: Merged Model (SLERP)": ID_COLOR}, ax=ax,
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=5, fontweight="bold")

ax.set_title(
    "Perbandingan Akurasi per Strata (Bahasa Indonesia, Cross-lingual)\nPure Model vs Merged Model",
    pad=15, fontweight="bold",
)

ax.set_ylim(0, 115)
ax.set_ylabel("Akurasi (%)")
ax.legend(title=None, loc="upper right", frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "comparison_pure_vs_merged_stratum_id.png"), dpi=300, bbox_inches="tight")
plt.show()



### Perbandingan Progres Akurasi per Epoch (Pure vs Merged, ID Cross-lingual)

In [ ]:
df_epoch_pure = pd.DataFrame(result_pure_model["epoch_records"]).sort_values("epoch")
df_epoch_merged = pd.DataFrame(result_merged_model["epoch_records"]).sort_values("epoch")

fig, ax = plt.subplots(figsize=(10, 6))

if not df_epoch_pure.empty:
    ax.plot(
        df_epoch_pure["epoch"], df_epoch_pure["accuracy_id_cross_val"] * 100,
        color=EN_COLOR, linewidth=2.5, marker="o", markersize=7, label="Pure Model (validation split)",
    )

if not df_epoch_merged.empty:
    ax.plot(
        df_epoch_merged["epoch"], df_epoch_merged["accuracy_id_cross_val"] * 100,
        color=ID_COLOR, linewidth=2.5, marker="o", markersize=7, label="Merged Model / SLERP (validation split)",
    )

ax.set_title(
    "Perbandingan Progres Akurasi ID Cross-lingual per Epoch\nPure Model vs Merged Model (validation split SFT)",
    pad=15, fontweight="bold",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Akurasi (%)")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.set_ylim(0, 100)
ax.legend(frameon=False, loc="lower right")
ax.yaxis.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "comparison_pure_vs_merged_epoch_progression.png"), dpi=300, bbox_inches="tight")
plt.show()

## Section 14 - Ringkasan & Penyimpanan Hasil Gabungan (Pure vs Merged)

In [ ]:
def build_summary_row(result, base_model_path):
    row = {
        "run_name": result["run_name"],
        "base_model": base_model_path,
        "target_cot_col": result["target_cot_col"],
        "accuracy_en": result["accuracy_en"],
        "accuracy_id_cross": result["accuracy_id_cross"],
    }
    row.update({f"stratum_en_{k}": v for k, v in result["per_stratum_en"].items()})
    row.update({f"stratum_id_cross_{k}": v for k, v in result["per_stratum_id_cross"].items()})
    return row

In [ ]:
summary_rows = [
    build_summary_row(result_pure_model, MODEL_MATH),
    build_summary_row(result_merged_model, SLERP_MERGED_DIR),
]

df_combined_summary = pd.DataFrame(summary_rows)
df_combined_summary.to_csv(COMBINED_SFT_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print(f"\nRingkasan gabungan (Pure vs Merged) disimpan di: {COMBINED_SFT_SUMMARY_PATH}")
print(df_combined_summary.to_string(index=False))

print("\n=== Selisih Akurasi (Merged - Pure) ===")
print(f"  Delta EN         : {(result_merged_model['accuracy_en'] - result_pure_model['accuracy_en']) * 100:+.2f} poin")
print(f"  Delta ID (cross)  : {(result_merged_model['accuracy_id_cross'] - result_pure_model['accuracy_id_cross']) * 100:+.2f} poin")